# Getting the Bronze Data From External storage

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
base_path = 'abfss://geolocation@adlsstdout001tgt.dfs.core.windows.net/'

In [0]:
from pyspark.sql.functions import current_date, lit

base_path = "abfss://geolocation@adlsstdout001tgt.dfs.core.windows.net/geolocation/"

# ---- Year ----
folders = dbutils.fs.ls(base_path)
latest_year = max([f.name.replace('/', '') for f in folders])

# ---- Month ----
folders = dbutils.fs.ls(f"{base_path}{latest_year}/")
latest_month = max([f.name.replace('/', '') for f in folders])

# ---- Day ----
folders = dbutils.fs.ls(f"{base_path}{latest_year}/{latest_month}/")
latest_day = max([f.name.replace('/', '') for f in folders])

# ---- Final Path ----
latest_path = f"{base_path}{latest_year}/{latest_month}/{latest_day}/*.parquet"

print(f"Reading from: {latest_path}")
# Construct file_date from folder
file_date = f"{latest_year}-{latest_month}-{latest_day}"


# ---- Read parquet ----
raw_df = (
    spark.read
    .format("parquet")
    .load(latest_path)
    .withColumn("ingest_date", current_date())
    .withColumn("file_date", lit(file_date))
                 
)

display(raw_df.limit(10))


In [0]:
from pyspark.sql.window import Window
# =========================================================
# INDUSTRY LEVEL DATA CLEANING
# =========================================================

df_cleaned = (
    raw_df

    # -----------------------------------------------------
    # STANDARDIZE COLUMN NAMES
    # -----------------------------------------------------
    .select(
        F.col("geolocation_zip_code_prefix").cast("string"),
        F.col("geolocation_lat").cast("double"),
        F.col("geolocation_lng").cast("double"),
        F.col("geolocation_city").cast("string"),
        F.col("geolocation_state").cast("string"),
        F.col("ingest_date").cast("date"),
        F.col("file_date").cast("string")
    )

    # -----------------------------------------------------
    # REMOVE LEADING/TRAILING SPACES
    # -----------------------------------------------------
    .withColumn(
        "geolocation_zip_code_prefix",
        F.trim(F.col("geolocation_zip_code_prefix"))
    )
    .withColumn(
        "geolocation_city",
        F.trim(F.col("geolocation_city"))
    )
    .withColumn(
        "geolocation_state",
        F.trim(F.col("geolocation_state"))
    )

    # -----------------------------------------------------
    # STANDARDIZE TEXT
    # -----------------------------------------------------
    .withColumn(
        "geolocation_city",
        F.initcap(
            F.translate(
                F.lower(F.col("geolocation_city")),
                "áàãâäéèêëíìîïóòõôöúùûüç",
                "aaaaaeeeeiiiiooooouuuuc"
            )
        )
    )

    .withColumn(
        "geolocation_state",
        F.upper(F.col("geolocation_state"))
    )

    # -----------------------------------------------------
    # REMOVE INVALID ZIP CODES
    # BRAZIL ZIP PREFIX = 5 DIGITS
    # -----------------------------------------------------
    .filter(
        F.col("geolocation_zip_code_prefix").rlike("^[0-9]{5}$")
    )

    # -----------------------------------------------------
    # REMOVE NULL CRITICAL VALUES
    # -----------------------------------------------------
    .filter(
        F.col("geolocation_lat").isNotNull()
    )
    .filter(
        F.col("geolocation_lng").isNotNull()
    )
    .filter(
        F.col("geolocation_city").isNotNull()
    )
    .filter(
        F.col("geolocation_state").isNotNull()
    )

    # -----------------------------------------------------
    # VALIDATE LATITUDE / LONGITUDE RANGES
    # -----------------------------------------------------
    .filter(
        (F.col("geolocation_lat").between(-90, 90))
        &
        (F.col("geolocation_lng").between(-180, 180))
    )

    # -----------------------------------------------------
    # REMOVE DUPLICATES
    # -----------------------------------------------------
    .dropDuplicates([
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ])
)

# =========================================================
# OPTIONAL:
# KEEP ONLY ONE RECORD PER ZIP CODE
# (BEST PRACTICE FOR DIMENSION STYLE TABLE)
# =========================================================

window_spec = Window.partitionBy("geolocation_zip_code_prefix") \
                    .orderBy(F.col("geolocation_city"))

df_final = (
    df_cleaned
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# =========================================================
# WRITE TO SILVER LAYER
# =========================================================

(
        df_final.write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .option(
                    "path",
                    "abfss://silver@silverprocessdbstorage.dfs.core.windows.net/GeoLocation"
                )
            .saveAsTable("ecom.silver.dim_silver_geo")
    )

print("Geolocation silver table loaded successfully")